In [ ]:
#transformer总体架构
#输入：文本序列
#输出：文本序列
#编码器部分
#解码器部分

#输入部分包含词嵌入层和位置编码层
#词嵌入层将输入的文本序列转换为向量表示
#位置编码层为输入的文本序列添加位置信息，使模型能够捕捉到序列中单词之间的相对位置关系

#编码器部分包含多个编码器层，每个编码器层由多头自注意力机制和前馈神经网络组成
#每个编码器层由两个子层组成：多头自注意力机制和前馈神经网络
#第一个子层是多头自注意力机制以及一个残差连接，它允许模型在不同的表示子空间中关注输入序列的不同部分，从而捕捉到输入序列中的长距离依赖关系，残差连接有助于缓解深层网络中的梯度消失问题
#第二个子层包括一个前馈神经网络和规范化层以及一个残差连接，前馈神经网络由两个线性变换和一个激活函数组成，规范化层用于稳定训练过程，残差连接有助于缓解深层网络中的梯度消失问题

#解码器部分也包含多个解码器层，每个编码器层由三个子层组成：多头自注意力机制、编码器-解码器注意力机制和前馈神经网络，第一个子层为带掩码的多头自注意力机制以及残差连接和规范化层，第二个子层为多头自注意力机制层以及残差连接和规范化层，第三个子层为前馈神经网络层以及残差连接和规范化层

In [ ]:
#文本嵌入层
#文本嵌入层将输入的文本序列转换为向量表示
#注意：在文本嵌入时，经常在embedding后乘以一个缩放因子，通常是嵌入维度的平方根。这是因为在Transformer中，输入的嵌入向量会被送入多头自注意力机制中进行计算，而多头自注意力机制中的点积操作可能会导致数值不稳定。通过乘以缩放因子，可以使得输入的嵌入向量的数值范围更适合进行点积计算，从而提高模型的训练稳定性和性能。

#文本嵌入层实现
import torch.nn as nn
import torch
import math
embedding=nn.Embedding(100,10,padding_idx=0) #词表大小为100，嵌入维度为10，padding_idx=0表示将索引为0的词视为填充词，不进行训练
input=torch.tensor([[1,2,3,4,0],[5,6,7,8,0]],dtype=torch.long) #输入的文本序列，包含两个样本，每个样本包含5个词，其中索引为0的词表示填充词
output=embedding(input)
print(output)

class Embedding(nn.Module):
    def __init__(self,vocab_size,embedding_dim):
        super().__init__()
        self.vocab_size=vocab_size
        self.embedding_dim=embedding_dim
        self.embed=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)
    def forward(self,x):
        x=self.embed(x)
        return x*math.sqrt(self.embedding_dim) #乘以缩放因子

embedding=Embedding(100,20)
output=embedding(input)
print(output)

tensor([[[-1.3360e+00,  2.6282e-01,  8.7153e-01, -2.9138e-01, -4.9675e-01,
          -5.7583e-02, -6.7785e-01, -3.5831e-01,  1.7028e+00, -2.6191e-01],
         [ 1.4189e+00, -5.3983e-01, -1.1192e+00,  3.1052e+00, -2.9904e-01,
           7.8256e-03,  3.3412e-01, -9.8451e-01,  6.9133e-02,  7.1807e-02],
         [-9.2751e-01, -1.4812e+00,  1.2208e+00,  6.4810e-01, -4.0849e-01,
          -2.1597e-01,  8.6345e-01,  9.9385e-01,  4.2135e-01,  1.7414e-01],
         [ 4.0660e-01,  7.7001e-01,  3.9010e-01,  3.2093e-01,  5.7832e-01,
           1.7955e+00,  1.6043e+00,  1.7365e+00, -4.4839e-01,  1.2742e-01],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[-7.4122e-01,  5.3662e-01, -2.2253e+00,  6.0479e-01,  1.8204e+00,
           1.2902e+00, -2.4590e-01,  1.2322e-01, -3.8393e-01, -1.0054e+00],
         [ 3.8751e-01,  6.9070e-01,  4.5428e-01, -1.8804e+00,  5.1951e-01,
          -2.7242

In [5]:
#位置编码层
#位置编码层为输入的文本序列添加位置信息，使模型能够捕捉到序列中单词之间的相对位置关系，将索引转化为位置编码向量
#三角函数位置编码公式
#PE(pos,2i)=sin(pos/10000^(2i/d_model)) 奇数位使用正弦函数，偶数位使用余弦函数
#PE(pos,2i+1)=cos(pos/10000^(2i/d_model))
#其中pos表示单词在序列中的位置，i表示嵌入维度的索引，d_model表示嵌入维度的大小

#位置编码层的实现
class PositionalEncoding(nn.Module):
    def __init__(self,dropout,d_model,max_len):
        super().__init__()
        self.dropout=nn.Dropout(dropout) #dropout层用于防止过拟合

        pe=torch.zeros(max_len,d_model) #初始化pe矩阵，大小为(max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小
        position=torch.arange(0,max_len,dtype=torch.long).unsqueeze(1) #生成位置索引，大小为(max_len, 1)，其中max_len表示最大序列长度
        div_term=torch.exp(torch.arange(0,max_len,dtype=torch.long)*-(math.log(10000)/d_model)) #计算分母项，大小为(d_model/2)，其中d_model表示嵌入维度的大小

        #position和div_term相乘得到一个大小为(max_len, d_model/2)的矩阵，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

         #计算位置编码矩阵，使用三角函数公式，奇数位使用正弦函数，偶数位使用余弦函数
         #pe矩阵的大小为(max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

        pe[:,0::2]=torch.sin(position*div_term)
        pe[:,1::2]=torch.cos(position*div_term)
        pe=pe.unsqueeze(0) #添加维度，与词嵌入矩阵维度匹配，大小为(1, max_len, d_model)，其中max_len表示最大序列长度，d_model表示嵌入维度的大小

        self.register_buffer('pe',pe) #将pe矩阵注册为模型的缓冲区，使其在模型保存和加载时能够正确处理，并且在训练过程中不会被更新
    def forward(self,x):
        x=x+self.pe[:,x.size(1)]
        return self.dropout(x)



In [ ]:
#掩码张量的介绍
#掩码张量是一个布尔类型的张量，用于指示模型在计算注意力权重时应该忽略哪些位置。掩码张量的形状通常为(batch_size, seq_len)，其中batch_size表示批次大小，seq_len表示序列长度。掩码张量中的值为True表示对应位置应该被忽略，值为False表示对应位置应该被考虑。
#在Transformer模型中，掩码张量主要用于两种情况：填充掩码和未来掩码。填充掩码用于指示模型在计算注意力权重时应该忽略填充位置，未来掩码用于指示模型在计算注意力权重时应该忽略未来位置，以防止模型在训练过程中看到未来的信息。

#transformer中自注意力机制的实现
#自注意力机制允许模型在不同的表示子空间中关注输入序列的不同部分，从而捕捉到输入序列中的长距离依赖关系。自注意力机制的核心是计算注意力权重，这些权重表示输入序列中每个位置对其他位置的关注程度。注意力权重的计算通常使用点积操作，具体来说，给定一个输入序列的表示矩阵X，首先通过线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V，然后计算注意力权重矩阵A，A=softmax(QK^T/sqrt(d_k))，其中d_k是键矩阵的维度。最后，通过将注意力权重矩阵A与值矩阵V相乘，得到自注意力机制的输出矩阵O，O=AV。自注意力机制的输出矩阵O包含了输入序列中每个位置对其他位置的关注信息，从而使模型能够捕捉到输入序列中的长距离依赖关系。

#自注意力机制中的Q,K,V矩阵的计算
#给定一个输入序列的表示矩阵X，首先通过线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V。具体来说，给定一个输入序列的表示矩阵X，首先通过三个不同的线性变换将其映射到查询矩阵Q、键矩阵K和值矩阵V。线性变换的参数通常是可学习的权重矩阵W_Q、W_K和W_V，分别用于计算查询矩阵Q、键矩阵K和值矩阵V。

In [9]:
#生成掩码向量

#测试生成上三角矩阵
import numpy as np
print(np.triu([[1,1,1],[0,1,1],[0,0,1]],k=1)) #生成一个上三角矩阵，k=1表示主对角线以上的元素保留，主对角线及以下的元素为0
print(np.triu([[1,1,1],[0,1,1],[0,0,1]],k=0)) #生成一个上三角矩阵，k=0表示主对角线以上的元素保留，主对角线以下的元素为0

size=5
mask=torch.from_numpy(1-np.triu(np.ones((1,size,size)),k=1).astype(int)) #生成一个掩码举着呢，三维，np.triu生成一个上三角矩阵，k=1表示主对角线以上的元素保留，主对角线及以下的元素为0，np.ones((1,size,size))生成一个全1的三维矩阵，1-np.triu(...)将上三角矩阵中的1变为0，0变为1，最后使用torch.from_numpy将其转换为PyTorch张量
print(mask)

[[0 1 1]
 [0 0 1]
 [0 0 0]]
[[1 1 1]
 [0 1 1]
 [0 0 1]]
tensor([[[1, 0, 0, 0, 0],
         [1, 1, 0, 0, 0],
         [1, 1, 1, 0, 0],
         [1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1]]], dtype=torch.int32)


In [ ]:
#sentence_mask
#生成一个句子掩码，用于指示模型在计算注意力权重时应该忽略哪些位置。句子掩码通常用于指示模型在计算注意力权重时应该忽略填充位置，以防止模型在训练过程中看到填充的信息。

#例如，给定一个输入序列的表示矩阵X和一个对应的句子掩码mask，模型在计算注意力权重时会将mask中为True的位置对应的注意力权重设置为负无穷，从而使得这些位置在计算注意力权重时被忽略。


In [14]:
#sentence_mask演示
torch.manual_seed(42)
input=torch.randn(1,size,size)
mask=torch.from_numpy(1-np.triu(np.ones((1,size,size)),k=1).astype(int))
print(mask)
print(input)
input.masked_fill_(mask==0,float('-inf')) #使用掩码填充的方法将mask中为0的位置赋值为负无穷，最后使得这些位置在计算权重的时候被忽略

tensor([[[1, 0, 0, 0, 0],
         [1, 1, 0, 0, 0],
         [1, 1, 1, 0, 0],
         [1, 1, 1, 1, 0],
         [1, 1, 1, 1, 1]]], dtype=torch.int32)
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
         [-1.2345, -0.0431, -1.6047, -0.7521, -0.6866],
         [-0.4934,  0.2415, -1.1109,  0.0915, -2.3169],
         [-0.2168, -1.3847, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]]])


tensor([[[ 1.9269,    -inf,    -inf,    -inf,    -inf],
         [-1.2345, -0.0431,    -inf,    -inf,    -inf],
         [-0.4934,  0.2415, -1.1109,    -inf,    -inf],
         [-0.2168, -1.3847, -0.3957,  0.8034,    -inf],
         [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]]])

In [ ]:
#padding_mask的原理